In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

## Dictionary
This example uses stardict for dictionary. You can also use other dictionary formats, like a plain json. If you choose to do so, you must reimplement the function below

In [2]:
from pystardict import Dictionary
dict1 = Dictionary("dictionaries/stardict-es-en_Babylon-2.4.2/Spanish-English_Babylon")
def get_def(word):
    try:
        return dict1[word]
    except KeyError:
        return None

In [3]:
# Try look up a word
get_def("palabra")

'<font color="#007000">feminine noun</font> palabra <br>\nword, unit of language with meaning; term; speech'

## Language Model
Choose a language here. Note that only some languages are supported by SpaCy.

Check https://spacy.io/usage/models for model names.

In [4]:
TL = "es"

In [5]:
import spacy
nlp = spacy.load(TL + '_core_news_lg', disable=['ner', 'parser'])

## Load CSV file from Tatoeba
__IMPORTANT__: You need to check if they are in the right order. You may have to swap the order of "en" and "tl" tags.

In [6]:
d = pd.read_csv('sentences/en_' + TL + '.tsv', sep='\t', header=None, names=['id1', 'en', 'id2', 'tl'])

In [7]:
d = d[['en', 'tl']]

In [8]:
sample = d[:10000]
## Make sure this looks right
sample

,en,tl
0,Let's try something.,¡Intentemos algo!
1,I have to go to sleep.,Tengo que irme a dormir.
2,Today is June 18th and it is Muiriel's birthday!,¡Hoy es 18 de junio y es el cumpleaños de Muir...
3,Today is June 18th and it is Muiriel's birthday!,¡Hoy es el 18 de junio y es el cumpleaños de M...
4,Muiriel is 20 now.,"Ahora, Muiriel tiene 20 años."
...,...,...
9995,"Brent is an American, but he speaks Japanese a...","Brent es americano, pero habla japonés como si..."
9996,Have you heard from Freddie?,¿Has oído de Freddie?
9997,Have you heard from Freddie?,¿Sabes algo de Freddie?
9998,Fred wrote his mother a long letter.,Fred escribió una extensa carta a su madre.


In [9]:
def remove_punctuations(l):
    return [item for item in l if item.isalpha()]

In [10]:
def getword(freq):
    return fl.loc[freq]['Lemma']

## Frequency List
This must be processed and be fully lemmatized, or else the result might be poor.
Open the one of the files under freqlists/ to see what it should look like.

This list must not contain any duplicates.

In [11]:
fl = pd.read_csv("freqlists/" + TL + ".csv", index_col=0)
fl2 = fl.reset_index().set_index("Lemma")
# Make sure this looks right
fl.head(10)

,Lemma
1,el
2,de
3,que
4,en
5,y
6,a
7,uno
8,ser
9,del
10,se


In [12]:
def test_coverage2(data, upto=2000, downto=200):
    vocab = set.union(*data['lemmas'].apply(set).tolist())
    words = set(fl.iloc[downto:upto]['Lemma'])
    inter = vocab.intersection(words)
    covered_ratio = len(inter) / (upto - downto)
    return covered_ratio

In [13]:
def get_missings(data, upto=5000, downto=200):
    vocab = set.union(*data['lemmas'].apply(set).tolist())
    words = set(fl.iloc[downto:upto]['Lemma'])
    inter = vocab.intersection(words)
    return words - inter

In [14]:
def format_lemmas(lemmas):
    return " · ".join(lemmas)

In [15]:
def get_dif3(l):
    if len(res:=fl2.reindex(l)['index'].fillna(0)) != 0:
        return int(max(res))
    else:
        return 0

In [16]:
def process_sentences2(data):
    print("STEP 1: Lemmatization")
    data.loc[:, "lemmas"] = [[token.lemma_ for token in doc] for doc in nlp.pipe(data['tl'].apply(str.lower), n_process=-1)]
    data.loc[:, 'lemmas'] = data['lemmas'].apply(remove_punctuations)
    print("STEP 2: Calculating frequency scores")
    data.loc[:, 'difficulty'] = data['lemmas'].apply(get_dif3)
    data.loc[:, 'lemmas_formatted'] = data['lemmas'].apply(format_lemmas)
    data = data[data['difficulty'] < 15000]
    return data.sort_values('difficulty')

In [17]:
def test_cov_levels(data):
    a = [2000, 5000, 10000, 15000]
    b = [format(100 * test_coverage2(data, level), "2.1f") + "%" for level in a]
    return pd.DataFrame({"Level": a, "Coverage": b})

In [18]:
def get_complemented(processed):
    print("STEP 3: Deduplication")
    print("\tSize of processed data: ", len(processed))
    norepeat = processed.drop_duplicates("difficulty")
    print("\tSize of deduplicated data: ", len(norepeat))
    print("\tCoverage of deduped data:")
    print(test_cov_levels(norepeat))
    missings = get_missings(norepeat) - get_missings(processed)
    result = pd.DataFrame()
    print("STEP 4: Finding complement")
    for word in missings:
        row = processed[processed.lemmas_formatted.str.contains(" " + word + " ")].reset_index(drop=True).reindex([0]).loc[0]
        row['target'] = word
        result = result.append(row)
    result = result[["en", "tl", "lemmas", "difficulty", "lemmas_formatted", "target"]]
    complement = result.dropna()
    enhanced = norepeat.append(complement)
    print("\tSize of enhanced data: ", len(enhanced))
    return enhanced.sort_values('difficulty').reset_index(drop=True)

In [19]:
def add_targets(data):
    print("STEP 5: Add targets to sentences")
    data = data[data.difficulty > 200]
    data.loc[data['target'].isna(), "target"] = data.loc[data['target'].isna(), "difficulty"].apply(getword)
    return data

In [20]:
def lookup_defs(data):
    print("STEP 6: Look up definitions")
    data.loc[:, "definition"] = data.target.apply(get_def)
    return data

In [21]:
def full_processing(data):
    output = process_sentences2(data)
    print(test_cov_levels(output))
    output = get_complemented(output)
    print(test_cov_levels(output))
    output = lookup_defs(add_targets(output))
    n_def = len(output.definition.notna())
    print(n_def, " definitions found.")
    return output

## Processing
Now all the relevant functions have been defined. We can start with a small sample to check for errors before running it on the full list.

In [22]:
#Calculate a sample first to check for errors. If this runs successfully, you may use the full list.
sm = full_processing(sample)

STEP 1: Lemmatization
STEP 2: Calculating frequency scores
   Level Coverage
0   2000    64.2%
1   5000    45.1%
2  10000    29.8%
3  15000    21.6%
STEP 3: Deduplication
	Size of processed data:  9112
	Size of deduplicated data:  2537
	Coverage of deduped data:
   Level Coverage
0   2000    56.2%
1   5000    40.3%
2  10000    27.2%
3  15000    19.9%
STEP 4: Finding complement
	Size of enhanced data:  2698
   Level Coverage
0   2000    61.8%
1   5000    43.8%
2  10000    28.8%
3  15000    21.0%
STEP 5: Add targets to sentences
STEP 6: Look up definitions
2605  definitions found.


In [ ]:
# The full processing. Will take some time.
full = full_processing(d)
full.to_csv("output.csv")